In [1]:
!python -m spacy download de_core_news_sm
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 95.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 104.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import spacy
import numpy as np

In [3]:
# --- 1. Load Spacy Tokenizers ---
spacy_ger = spacy.load("de_core_news_sm")
spacy_eng = spacy.load("en_core_web_sm")

def tokenize_ger(text):
    return [tok.text for tok in spacy_ger.tokenizer(text)]

def tokenize_eng(text):
    return [tok.text for tok in spacy_eng.tokenizer(text)]

In [4]:
# --- 2. Load Data ---
def load_data(de_path, en_path):
    with open(de_path, 'r', encoding='utf-8') as f_de, open(en_path, 'r', encoding='utf-8') as f_en:
        german_sentences = f_de.readlines()
        english_sentences = f_en.readlines()
    return german_sentences, english_sentences

train_de_path = "/kaggle/input/machine-translation-with-multi30k-de-en/.data/Multi30k/train.de"
train_en_path = "/kaggle/input/machine-translation-with-multi30k-de-en/.data/Multi30k/train.en"
train_de, train_en = load_data(train_de_path, train_en_path)


In [5]:
# --- 3. Tokenize and Build Vocab ---
tokenized_train_de = [['<sos>'] + tokenize_ger(sentence.lower().strip()) + ['<eos>'] for sentence in train_de]
tokenized_train_en = [['<sos>'] + tokenize_eng(sentence.lower().strip()) + ['<eos>'] for sentence in train_en]

from collections import Counter

def build_vocab(tokenized_sentences, max_size=10000, min_freq=2):
    counter = Counter()
    for sentence in tokenized_sentences:
        counter.update(sentence)

    vocab = {'<pad>': 0, '<unk>': 1}
    idx = 2
    for word, freq in counter.most_common(max_size):
        if freq >= min_freq:
            vocab[word] = idx
            idx += 1
    return vocab

vocab_de = build_vocab(tokenized_train_de)
vocab_en = build_vocab(tokenized_train_en)


In [6]:
# --- 4. Convert to Index Tensors ---
def numericalize(tokens, vocab):
    return [vocab.get(token, vocab['<unk>']) for token in tokens]

numericalized_train_de = [torch.tensor(numericalize(sentence, vocab_de)) for sentence in tokenized_train_de]
numericalized_train_en = [torch.tensor(numericalize(sentence, vocab_en)) for sentence in tokenized_train_en]


In [7]:
# --- 5. Dataset and DataLoader ---
class TranslationDataset(Dataset):
    def __init__(self, src_data, trg_data):
        self.src_data = src_data
        self.trg_data = trg_data

    def __len__(self):
        return len(self.src_data)

    def __getitem__(self, idx):
        return self.src_data[idx], self.trg_data[idx]

def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)
    src_batch = pad_sequence(src_batch, padding_value=vocab_de['<pad>'], batch_first=True)
    trg_batch = pad_sequence(trg_batch, padding_value=vocab_en['<pad>'], batch_first=True)
    return src_batch, trg_batch

dataset = TranslationDataset(numericalized_train_de, numericalized_train_en)
dataloader = DataLoader(dataset, batch_size=32, collate_fn=collate_fn, shuffle=True)

In [8]:
# --- 6. Transformer Model ---
class Transformer(nn.Module):
    def __init__(self, embedding_size, src_vocab_size, trg_vocab_size, src_pad_idx, num_heads, num_encoder_layers,
                 num_decoder_layers, forward_expansion, dropout, max_len, device):
        super(Transformer, self).__init__()
        self.src_word_embedding = nn.Embedding(src_vocab_size, embedding_size)
        self.src_position_embedding = nn.Embedding(max_len, embedding_size)
        self.trg_word_embedding = nn.Embedding(trg_vocab_size, embedding_size)
        self.trg_position_embedding = nn.Embedding(max_len, embedding_size)

        self.device = device
        self.transformer = nn.Transformer(d_model=embedding_size,
                                          nhead=num_heads,
                                          num_encoder_layers=num_encoder_layers,
                                          num_decoder_layers=num_decoder_layers,
                                          dim_feedforward=forward_expansion * embedding_size,
                                          dropout=dropout)

        self.fc_out = nn.Linear(embedding_size, trg_vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.src_pad_idx = src_pad_idx

    def make_src_mask(self, src):
        src_mask = (src == self.src_pad_idx).transpose(0, 1)
        return src_mask.to(self.device)

    def forward(self, src, trg):
        src_seq_length, N = src.shape
        trg_seq_length, N = trg.shape

        src_positions = torch.arange(0, src_seq_length).unsqueeze(1).expand(src_seq_length, N).to(self.device)
        trg_positions = torch.arange(0, trg_seq_length).unsqueeze(1).expand(trg_seq_length, N).to(self.device)

        embed_src = self.dropout(self.src_word_embedding(src) + self.src_position_embedding(src_positions))
        embed_trg = self.dropout(self.trg_word_embedding(trg) + self.trg_position_embedding(trg_positions))

        src_padding_mask = self.make_src_mask(src)
        trg_mask = self.transformer.generate_square_subsequent_mask(trg_seq_length).to(self.device)

        out = self.transformer(embed_src, embed_trg, src_key_padding_mask=src_padding_mask, tgt_mask=trg_mask)
        return self.fc_out(out)

In [9]:
# --- 7. Training Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

embedding_size = 512
num_heads = 8
num_encoder_layers = 3
num_decoder_layers = 3
dropout = 0.1
max_len = 100
forward_expansion = 4
src_pad_idx = vocab_de['<pad>']

src_vocab_size = len(vocab_de)
trg_vocab_size = len(vocab_en)

model = Transformer(embedding_size, src_vocab_size, trg_vocab_size, src_pad_idx,
                    num_heads, num_encoder_layers, num_decoder_layers,
                    forward_expansion, dropout, max_len, device).to(device)

optimizer = optim.Adam(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss(ignore_index=vocab_en['<pad>'])

/usr/local/lib/python3.10/dist-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [10]:
# --- 8. Training Loop ---
num_epochs = 20

for epoch in range(num_epochs):
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    model.train()
    epoch_loss = 0

    for batch_idx, (src, trg) in enumerate(dataloader):
        src = src.transpose(0, 1).to(device)
        trg = trg.transpose(0, 1).to(device)

        output = model(src, trg[:-1, :])
        output = output.reshape(-1, output.shape[2])
        trg = trg[1:].reshape(-1)

        optimizer.zero_grad()
        loss = criterion(output, trg)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(dataloader)
    print(f"Average Loss: {avg_loss:.4f}")

Epoch [1/20]
Average Loss: 3.5967
Epoch [2/20]
Average Loss: 2.5885
Epoch [3/20]
Average Loss: 2.1379
Epoch [4/20]
Average Loss: 1.8484
Epoch [5/20]
Average Loss: 1.6308
Epoch [6/20]
Average Loss: 1.4545
Epoch [7/20]
Average Loss: 1.3099
Epoch [8/20]
Average Loss: 1.1832
Epoch [9/20]
Average Loss: 1.0714
Epoch [10/20]
Average Loss: 0.9748
Epoch [11/20]
Average Loss: 0.8898
Epoch [12/20]
Average Loss: 0.8163
Epoch [13/20]
Average Loss: 0.7501
Epoch [14/20]
Average Loss: 0.6963
Epoch [15/20]
Average Loss: 0.6464
Epoch [16/20]
Average Loss: 0.6030
Epoch [17/20]
Average Loss: 0.5650
Epoch [18/20]
Average Loss: 0.5337
Epoch [19/20]
Average Loss: 0.5036
Epoch [20/20]
Average Loss: 0.4771


In [11]:
# --- 9. Translation Function ---
def translate_sentence(model, sentence, vocab_src, vocab_trg, device, max_length=50):
    model.eval()
    tokens = ['<sos>'] + tokenize_ger(sentence.lower().strip()) + ['<eos>']
    numericalized = [vocab_src.get(token, vocab_src['<unk>']) for token in tokens]
    src_tensor = torch.tensor(numericalized).unsqueeze(1).to(device)

    src_mask = model.make_src_mask(src_tensor)
    memory = model.transformer.encoder(model.src_word_embedding(src_tensor) + 
                                       model.src_position_embedding(torch.arange(0, src_tensor.shape[0]).unsqueeze(1).to(device)),
                                       src_key_padding_mask=src_mask)

    outputs = [vocab_trg['<sos>']]
    for _ in range(max_length):
        trg_tensor = torch.tensor(outputs).unsqueeze(1).to(device)
        trg_mask = model.transformer.generate_square_subsequent_mask(len(trg_tensor)).to(device)

        with torch.no_grad():
            output = model.transformer.decoder(model.trg_word_embedding(trg_tensor) + 
                                               model.trg_position_embedding(torch.arange(0, trg_tensor.shape[0]).unsqueeze(1).to(device)),
                                               memory,
                                               tgt_mask=trg_mask)
        output = model.fc_out(output)
        best_guess = output.argmax(2)[-1, :].item()

        outputs.append(best_guess)

        if best_guess == vocab_trg['<eos>']:
            break

    translated_sentence = [k for k, v in vocab_trg.items() if v in outputs]
    return ' '.join(translated_sentence[1:-1])

In [12]:
# --- 10. Example Translation ---
example_sentence = "komm auf die dunkle Seite, wir haben Kekse."
translation = translate_sentence(model, example_sentence, vocab_de, vocab_en, device)
print(f"Translated Sentence: {translation}")


Translated Sentence: <eos> . the on of side dark all line color finish
